In [9]:
import polars as pl
train_data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [10]:
import importlib
import ml_pipeline
import mlflow

importlib.reload(ml_pipeline)

/usr/local/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


<module 'ml_pipeline' from '/work/ml_pipeline.py'>

In [11]:
#mlflow.set_tracking_uri("http://mlflow:5000")
#mlflow.set_experiment("exp2")

In [12]:
from ml_pipeline import MoisturePipeline, FullPipelineModel

pipe = MoisturePipeline(
    use_pca=False,
    use_diff=True,
    params={
        "verbosity": -1,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "num_leaves": 64,
        "min_data_in_leaf": 5,
        "n_jobs": -1,
    }
)
rmse = pipe.fit(train_data_pl)

with mlflow.start_run():
    mlflow.log_params(pipe.params)
    #mlflow.log_param("pca_components", pipe.fe.pca.n_components)

    mlflow.log_metric("rmse", rmse)

    mlflow.pyfunc.log_model(
        name="model",
        python_model=FullPipelineModel(pipe)
    )

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/07 22:12:39 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run overjoyed-roo-226 at: http://mlflow:5000/#/experiments/0/runs/213369b820704aa9a6f212bff47c8a2f
🧪 View experiment at: http://mlflow:5000/#/experiments/0


### sharpray値を使い　2次微分は1,特定の特徴量だけが落としている　or 2.全体的に効いていない　をみる

In [13]:
# transform後のデータ
X = pipe.fe.transform(train_data_pl)

# 特徴量カラム（学習に使ったもの）
feature_cols = [c for c in X.columns if c not in pipe.drop_cols]

X = X.select(feature_cols)
y = train_data_pl["含水率"]  # ← 目的変数に合わせる

In [17]:
feature_cols.head() #show del

AttributeError: 'list' object has no attribute 'head'

In [24]:
def calc_sharpe_from_cols(cols):
        # 数値列だけ抽出
    numeric_cols = [
        c for c in cols
        if X[c].dtype.is_numeric()
    ]
    
    X_sub = X[cols]
    y_pred = pipe.model.predict(X_sub)

    returns = y_pred  # or y_pred - y

    return np.mean(returns) / np.std(returns)

In [25]:
from ml_pipeline import FeatureEngineer  # 定義している場所からimport

fe = FeatureEngineer()
fe.fit(train_data_pl)

base_cols = fe.base_cols

In [34]:
base_cols[:10]

['sample number',
 'species number',
 '含水率',
 '9993.76781',
 '9989.9107',
 '9986.05359',
 '9982.19648',
 '9978.33937',
 '9974.48227',
 '9970.62516']

In [26]:
calc_sharpe_from_cols(base_cols)

ColumnNotFoundError: "sample number" not found


## モデルを引き落としてきて実験する

In [75]:
import mlflow

experiment_name = "exp2"  # もしくはあなたの実験名

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1
)

latest_run_id = runs.iloc[0]["run_id"]
print(latest_run_id)

d1e10136841f4842b13694e6af97a4d5


In [76]:
model = mlflow.pyfunc.load_model(f"runs:/{latest_run_id}/model")

In [77]:
mlflow.pyfunc.log_model(
    name="model",
    python_model=FullPipelineModel(pipe)
)

2026/04/04 21:35:53 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


In [79]:
model

mlflow.pyfunc.loaded_model:
  artifact_path: /mlruns/5/models/m-278c1a7af9f2414f8926ccc85351266f/artifacts
  flavor: mlflow.pyfunc.model
  run_id: d1e10136841f4842b13694e6af97a4d5